# Cross-Network Prognosis Leaderboard

Walks all `checkpoints_prognoser_{combo}/` directories, parses `run_summary.json`,
and ranks (network_combo × method × feature_set × embedding_strategy) by test C-index,
integrated Brier score, and time-dependent AUC.

Run after sweeping `PROGNOSER_RUNNER.ipynb` over multiple combos × methods.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path('/mnt/e/fyassine/ad-early-detection')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
PROGNOSER_DIR = REPO_ROOT / 'PROGNOSER' / 'notebooks'
ARTIFACTS = PROGNOSER_DIR / '_artifacts_'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

In [ ]:
def collect_runs(prognoser_dir: Path) -> pd.DataFrame:
    rows = []
    for combo_dir in sorted(prognoser_dir.glob('checkpoints_prognoser_*')):
        if not combo_dir.is_dir():
            continue
        combo = combo_dir.name.replace('checkpoints_prognoser_', '')
        for run_dir in sorted(combo_dir.iterdir()):
            summary = run_dir / 'run_summary.json'
            if not summary.exists():
                continue
            with open(summary) as f:
                s = json.load(f)
            metrics = s.get('metrics', {})
            test = metrics.get('test', {})
            val  = metrics.get('val',  {})
            train = metrics.get('train', {})
            test_auc = test.get('auc', {})
            rows.append({
                'network_combo': combo,
                'method': s.get('method', 'unknown'),
                'feature_set': s.get('feature_set', ''),
                'embedding_strategy': s.get('embedding_strategy', 'baseline'),
                'n_features': s.get('n_features'),
                'n_train': s.get('n_train'), 'n_test': s.get('n_test'),
                'n_events_test': s.get('n_events_test'),
                'c_train': train.get('c_index'),
                'c_val':   val.get('c_index'),
                'c_test':  test.get('c_index'),
                'ibs_test': test.get('ibs'),
                'auc_test_24': test_auc.get('24'),
                'auc_test_36': test_auc.get('36'),
                'auc_test_60': test_auc.get('60'),
                'run_name': s.get('run_name', run_dir.name),
                'timestamp': s.get('timestamp', ''),
            })
    return pd.DataFrame(rows)

leaderboard = collect_runs(PROGNOSER_DIR)
if leaderboard.empty:
    print('No runs found. Run PROGNOSER_RUNNER.ipynb first.')
else:
    leaderboard = leaderboard.sort_values(['c_test', 'auc_test_36'], ascending=False).reset_index(drop=True)
    print(f'Total runs: {len(leaderboard)}')
    print(f'Combos:    {sorted(leaderboard.network_combo.unique())}')
    print(f'Methods:   {sorted(leaderboard.method.unique())}')
    print(f'Strategies:{sorted(leaderboard.embedding_strategy.unique())}')
leaderboard

In [ ]:
if not leaderboard.empty:
    # Best run per (combo, method, embedding_strategy)
    group_cols = ['network_combo', 'method', 'embedding_strategy']
    best = leaderboard.loc[
        leaderboard.groupby(group_cols)['c_test'].idxmax()
    ].reset_index(drop=True)

    display_cols = ['network_combo', 'method', 'feature_set', 'embedding_strategy',
                    'n_features', 'c_train', 'c_val', 'c_test',
                    'ibs_test', 'auc_test_24', 'auc_test_36']
    print('Best per (combo, method, strategy):')
    print(best[display_cols].to_string(index=False))

    leaderboard.to_csv(ARTIFACTS / 'leaderboard_all_runs.csv', index=False)
    best.to_csv(ARTIFACTS / 'leaderboard_best_per_group.csv', index=False)

In [ ]:
if not leaderboard.empty:
    # Heatmap: network_combo × method, coloured by best test C-index
    pivot = best.pivot_table(index='network_combo', columns='method', values='c_test', aggfunc='max')

    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns)*2), max(5, len(pivot.index))), constrained_layout=True)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0.4, vmax=1.0,
                linewidths=0.5, ax=ax)
    ax.set_title('Test C-index: network_combo × method (best run per group)')
    ax.set_xlabel('Method')
    ax.set_ylabel('Network combo')
    plt.savefig(ARTIFACTS / 'leaderboard_heatmap.png', dpi=150)
    plt.show()

    # Bar chart: methods compared across combos
    fig2, ax2 = plt.subplots(figsize=(14, 6), constrained_layout=True)
    pivot.plot(kind='bar', ax=ax2, colormap='viridis', edgecolor='black', alpha=0.85)
    ax2.axhline(0.5, color='red', linestyle=':', alpha=0.6, label='chance')
    ax2.set_title('Test C-index by network combo × method')
    ax2.set_ylabel('Test C-index')
    ax2.set_ylim(0, 1)
    ax2.tick_params(axis='x', rotation=30)
    ax2.legend(loc='best', fontsize=8)
    plt.savefig(ARTIFACTS / 'leaderboard_barplot.png', dpi=150)
    plt.show()

In [ ]:
if not leaderboard.empty:
    winner = leaderboard.iloc[0]
    print('=' * 65)
    print('  TOP RUN BY TEST C-INDEX')
    print('=' * 65)
    for k in ['network_combo', 'method', 'feature_set', 'embedding_strategy',
              'n_features', 'c_train', 'c_val', 'c_test',
              'ibs_test', 'auc_test_24', 'auc_test_36', 'auc_test_60', 'run_name']:
        v = winner.get(k)
        print(f'  {k:>22}: {v:.4f}' if isinstance(v, float) else f'  {k:>22}: {v}')